# GPT-OSS detail centrality and retention

This notebook asks whether GPT-OSS summaries preferentially select central details from the original story, how that preference varies with source age, and how it differs among `younger`, `actual`, and `older` summaries.

It uses only the existing final-layer, last-token GPT-OSS reduction. It does not run the language model, generate embeddings, use PCA, model compression ratio, or model layers.

Definitions:

- **Original-detail centrality:** mean cosine similarity between one original detail and every other original detail in that full text.
- **Retained:** at least one summary detail selected that original detail as its top-1 GPT-OSS contextual-embedding match. Multiple summary details may select the same original detail.
- **Omitted:** no summary detail selected that original detail.
- **Centrality gap:** mean centrality of retained original details minus mean centrality of omitted original details, calculated separately for every summary.

The top-1 rule is forced and has no minimum similarity threshold. Results describe representational selection under this matching rule, not definitive semantic retention.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("STORY_FEATURES_ROOT", Path.cwd())).resolve()
if not (ROOT / "mechanistic_interpretability").exists():
    ROOT = ROOT.parent

ANALYSIS_DIR = Path(os.environ.get(
    "GPT_OSS_ANALYSIS_DIR",
    ROOT / "mechanistic_interpretability/outputs/02_representation_extraction/analysis/gpt_oss_20b",
))
INPUT_PATH = Path(os.environ.get(
    "GPT_OSS_SOURCE_DETAIL_TABLE",
    ANALYSIS_DIR / "original_detail_representation_coverage.csv.gz",
))
CONDITION_ORDER = ["younger", "actual", "older"]
AGE_CENTER = 8.0

print("Repository:", ROOT)
print("Input:", INPUT_PATH)

## Representation being analyzed

The underlying `last_token_vectors` are captured at the final token of each segmented detail while the complete record is passed through GPT-OSS. At final layer $L$, this is approximately

$$
z_i^{(L)} = h_{\operatorname{end}(d_i)}^{(L)}(d_1,\ldots,d_i).
$$

Because GPT-OSS is causal, this is the representation of detail $i$ conditioned on the story or summary **prefix through that detail**, not on later text.

## RQ1 — Are the existing matched-detail observations usable?

Validate the reduced table, restrict it to the three summary conditions, and check that each summary contains both retained and omitted original details.

In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {INPUT_PATH}. Set GPT_OSS_SOURCE_DETAIL_TABLE to the existing "
        "original_detail_representation_coverage.csv.gz file."
    )

details = pd.read_csv(INPUT_PATH, low_memory=False)
required = {
    "record_id", "story_key", "condition", "summary_id", "source_age",
    "original_detail_index", "original_centrality",
    "selected_by_any_summary_detail", "representation", "layer",
}
missing = sorted(required - set(details.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

details["source_age"] = pd.to_numeric(details["source_age"], errors="coerce")
details["original_centrality"] = pd.to_numeric(details["original_centrality"], errors="coerce")
retained_source = details["selected_by_any_summary_detail"]
if pd.api.types.is_bool_dtype(retained_source):
    details["retained"] = retained_source
else:
    details["retained"] = retained_source.astype(str).str.lower().map({"true": True, "false": False})
if details["retained"].isna().any():
    raise ValueError("Could not parse selected_by_any_summary_detail as Boolean.")
details = details[details["condition"].isin(CONDITION_ORDER)].copy()

print("Rows:", f"{len(details):,}")
print("Stories:", f"{details['story_key'].nunique():,}")
print("Summary records:", f"{details['record_id'].nunique():,}")
representations = sorted(details["representation"].dropna().unique())
layers = sorted(details["layer"].dropna().unique())
print("Representations:", representations)
print("Layers:", layers)
if representations != ["last"] or len(layers) != 1:
    raise ValueError("Expected one final-layer, last-token reduction table.")
display(details.groupby("condition", observed=True).agg(
    stories=("story_key", "nunique"),
    summaries=("record_id", "nunique"),
    original_detail_observations=("original_detail_index", "size"),
).reindex(CONDITION_ORDER))

In [ ]:
summary_keys = ["record_id", "story_key", "condition", "summary_id", "source_age"]
summary_rows = []

for identity, frame in details.groupby(summary_keys, sort=False, dropna=False):
    retained = frame.loc[frame["retained"], "original_centrality"].dropna()
    omitted = frame.loc[~frame["retained"], "original_centrality"].dropna()
    row = dict(zip(summary_keys, identity))
    row.update({
        "n_original_details": len(frame),
        "n_retained": len(retained),
        "n_omitted": len(omitted),
        "retention_rate": float(frame["retained"].mean()),
        "mean_centrality_retained": float(retained.mean()) if len(retained) else np.nan,
        "mean_centrality_omitted": float(omitted.mean()) if len(omitted) else np.nan,
    })
    row["centrality_gap"] = row["mean_centrality_retained"] - row["mean_centrality_omitted"]
    summary_rows.append(row)

summary_data = pd.DataFrame(summary_rows)
summary_data["condition"] = pd.Categorical(
    summary_data["condition"], categories=CONDITION_ORDER, ordered=True
)
summary_data["has_both_groups"] = (
    (summary_data["n_retained"] > 0) & (summary_data["n_omitted"] > 0)
)
model_data = summary_data.loc[
    summary_data["has_both_groups"]
    & summary_data["centrality_gap"].notna()
    & summary_data["source_age"].notna()
].copy()
model_data["source_age_centered"] = model_data["source_age"] - AGE_CENTER

display(summary_data.groupby("condition", observed=True).agg(
    summaries=("record_id", "size"),
    usable_summaries=("has_both_groups", "sum"),
    median_original_details=("n_original_details", "median"),
    median_retention_rate=("retention_rate", "median"),
).reindex(CONDITION_ORDER))

## RQ2 — Are retained original details more central than omitted original details overall?

The descriptive effect is the per-summary raw centrality gap in cosine units. Positive values mean the retained set is more central. This is not a standardized Cohen's d. The first histogram normalizes within summary condition. The following panels normalize separately within each source-age cell for actual, younger, and older summaries; sparse age cells therefore remain visible but should be interpreted cautiously. The complete-triplet section below reports paired Cohen's $d_z$ for within-story condition contrasts.

In [ ]:
overall = model_data["centrality_gap"].agg(["count", "mean", "std", "median"])
overall["se"] = overall["std"] / np.sqrt(overall["count"])
overall["ci95_low"] = overall["mean"] - 1.96 * overall["se"]
overall["ci95_high"] = overall["mean"] + 1.96 * overall["se"]
display(overall.to_frame("value"))

import matplotlib.colors as mcolors

gap_bins = np.histogram_bin_edges(model_data['centrality_gap'].dropna(), bins=45)
condition_colors = {'younger': '#59A14F', 'actual': '#4C78A8', 'older': '#E15759'}
fig, ax = plt.subplots(figsize=(8, 5))
for condition in CONDITION_ORDER:
    values = model_data.loc[
        model_data['condition'].eq(condition), 'centrality_gap'
    ].dropna().to_numpy()
    if not len(values):
        continue
    ax.hist(
        values, bins=gap_bins, weights=np.ones(len(values)) / len(values),
        histtype='step', linewidth=2, color=condition_colors[condition],
        label=f'{condition} (n={len(values):,})',
    )
ax.axvline(0, color="black", linewidth=1)
ax.set(
    xlabel="Retained − omitted original-detail centrality",
    ylabel="Proportion within condition per bin",
    title="Proportional distribution of centrality gaps by condition",
)
ax.legend(title='Summary condition')
plt.show()

finite_ages = model_data['source_age'].dropna().to_numpy(float)
age_min, age_max = float(finite_ages.min()), float(finite_ages.max())
age_norm = mcolors.Normalize(vmin=age_min, vmax=age_max)
age_cmap = plt.get_cmap('viridis')
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), sharex=True, sharey=True)
age_panel_order = ['actual', 'younger', 'older']
for ax, condition in zip(axes, age_panel_order):
    condition_frame = model_data.loc[model_data['condition'].eq(condition)]
    for source_age, age_frame in condition_frame.groupby('source_age', observed=True):
        values = age_frame['centrality_gap'].dropna().to_numpy()
        if not len(values):
            continue
        ax.hist(
            values, bins=gap_bins, weights=np.ones(len(values)) / len(values),
            histtype='step', linewidth=1.3, alpha=.85,
            color=age_cmap(age_norm(float(source_age))),
        )
    ax.axvline(0, color='black', linewidth=.8)
    ax.set_title(f'{condition} summaries')
    ax.set_xlabel('Retained − omitted centrality')
axes[0].set_ylabel('Proportion within source age per bin')
age_mapper = plt.cm.ScalarMappable(norm=age_norm, cmap=age_cmap)
age_mapper.set_array([])
fig.colorbar(age_mapper, ax=axes.tolist(), label='Source age', fraction=.025, pad=.02)
fig.suptitle('Age-specific proportional centrality-gap distributions')
plt.show()

## RQ3 — How does the retained–omitted centrality gap differ descriptively by summary condition?

In [ ]:
condition_summary = model_data.groupby(
    "condition", observed=True
)["centrality_gap"].agg(["count", "mean", "std", "median"]).reindex(CONDITION_ORDER)
condition_summary["se"] = condition_summary["std"] / np.sqrt(condition_summary["count"])
condition_summary["ci95_low"] = condition_summary["mean"] - 1.96 * condition_summary["se"]
condition_summary["ci95_high"] = condition_summary["mean"] + 1.96 * condition_summary["se"]
display(condition_summary)

fig, ax = plt.subplots(figsize=(7, 4))
plot_data = [
    model_data.loc[model_data["condition"].eq(condition), "centrality_gap"].to_numpy()
    for condition in CONDITION_ORDER
]
ax.boxplot(plot_data, labels=CONDITION_ORDER, showfliers=False)
ax.axhline(0, color="black", linewidth=1)
ax.set(
    xlabel="Summary condition",
    ylabel="Centrality gap",
    title="Centrality selection by summary condition",
)
plt.show()

## RQ4 — Within each summary condition, how does the centrality gap vary with source age?

These are observed age-cell means shown before regression. Sparse cells, especially for younger summaries, should not be overinterpreted.

In [ ]:
age_condition = model_data.groupby(
    ["condition", "source_age"], observed=True
).agg(
    summaries=("record_id", "size"),
    mean_gap=("centrality_gap", "mean"),
    sd_gap=("centrality_gap", "std"),
).reset_index()
age_condition["se_gap"] = age_condition["sd_gap"] / np.sqrt(age_condition["summaries"])
display(age_condition.sort_values(["condition", "source_age"]))

colors = {"younger": "#59A14F", "actual": "#4C78A8", "older": "#E15759"}
fig, ax = plt.subplots(figsize=(9, 5))
for condition in CONDITION_ORDER:
    frame = age_condition[
        age_condition["condition"].eq(condition)
    ].sort_values("source_age")
    ax.errorbar(
        frame["source_age"], frame["mean_gap"], yerr=1.96 * frame["se_gap"],
        marker="o", capsize=2, label=condition, color=colors[condition],
    )
ax.axhline(0, color="black", linewidth=1)
ax.set(
    xlabel="Source age",
    ylabel="Mean centrality gap",
    title="Observed centrality selection across source age",
)
ax.legend(title="Summary condition")
plt.show()

## RQ5 — Do retained and omitted centrality follow different age trajectories?

Reshape each summary to two rows: its mean retained-detail centrality and mean omitted-detail centrality. First fit the additive model

```r
mean_centrality ~ retention_status + source_age + condition
                 + (1 | story) + (1 | summary)
```

which assumes a common retention difference and age slope. Then fit the interaction model

```r
mean_centrality ~ retention_status * source_age * condition
                 + (1 | story) + (1 | summary)
```

Both are fitted with maximum likelihood and compared by a likelihood-ratio test. The additive model gives the simple average effects. If interactions improve fit, the separate trajectories reveal whether retained centrality, omitted centrality, or both change with age.

In [ ]:
import statsmodels.formula.api as smf
from patsy import dmatrix
from scipy.stats import chi2, norm

level_columns = ['record_id', 'story_key', 'condition', 'summary_id',
                 'source_age', 'source_age_centered']
retained_levels = model_data[level_columns + ['mean_centrality_retained']].rename(
    columns={'mean_centrality_retained': 'mean_centrality'}
).copy()
retained_levels['retention_status'] = 'retained'
omitted_levels = model_data[level_columns + ['mean_centrality_omitted']].rename(
    columns={'mean_centrality_omitted': 'mean_centrality'}
).copy()
omitted_levels['retention_status'] = 'omitted'
level_data = pd.concat([retained_levels, omitted_levels], ignore_index=True).dropna(
    subset=['mean_centrality', 'source_age_centered', 'story_key', 'record_id']
)
level_data['condition'] = pd.Categorical(
    level_data['condition'], categories=['actual', 'younger', 'older']
)
level_data['retention_status'] = pd.Categorical(
    level_data['retention_status'], categories=['omitted', 'retained']
)
display(level_data.groupby(['condition', 'retention_status'], observed=True).agg(
    rows=('mean_centrality', 'size'),
    stories=('story_key', 'nunique'),
    observed_mean=('mean_centrality', 'mean'),
).reindex(pd.MultiIndex.from_product(
    [CONDITION_ORDER, ['omitted', 'retained']], names=['condition', 'retention_status']
)))

level_additive_formula = (
    'mean_centrality ~ source_age_centered + '
    'C(condition, Treatment(reference="actual")) + '
    'C(retention_status, Treatment(reference="retained"))'
)
level_formula = (
    'mean_centrality ~ source_age_centered * '
    'C(condition, Treatment(reference="actual")) * '
    'C(retention_status, Treatment(reference="retained"))'
)

def fit_level_model(model_formula):
    model = smf.mixedlm(
        model_formula, data=level_data, groups=level_data['story_key'],
        re_formula='1', vc_formula={'summary': '0 + C(record_id)'},
    )
    return model.fit(reml=False, method='lbfgs')

def clean_fixed_effect_table(result):
    names = list(result.fe_params.index)
    confidence = result.conf_int().loc[names]
    table = pd.DataFrame({
        'term': names,
        'estimate': result.fe_params.loc[names].to_numpy(float),
        'std_error': result.bse.loc[names].to_numpy(float),
        'z': result.tvalues.loc[names].to_numpy(float),
        'p_value': result.pvalues.loc[names].to_numpy(float),
        'ci95_low': confidence[0].to_numpy(float),
        'ci95_high': confidence[1].to_numpy(float),
    })
    table['significant_05'] = table.p_value < 0.05
    return table

level_additive_fit = fit_level_model(level_additive_formula)
print('ADDITIVE MODEL: common age and retention effects')
print(level_additive_fit.summary())
level_additive_table = clean_fixed_effect_table(level_additive_fit)
display(level_additive_table.round(4))

level_fit = fit_level_model(level_formula)
lr_statistic = max(0.0, 2 * (level_fit.llf - level_additive_fit.llf))
lr_df = len(level_fit.fe_params) - len(level_additive_fit.fe_params)
level_model_comparison = pd.DataFrame([
    {'model': 'additive', 'fixed_effects': len(level_additive_fit.fe_params),
     'log_likelihood': level_additive_fit.llf, 'aic': level_additive_fit.aic,
     'bic': level_additive_fit.bic, 'lr_vs_additive': np.nan, 'df': np.nan, 'p_value': np.nan},
    {'model': 'three-way interaction', 'fixed_effects': len(level_fit.fe_params),
     'log_likelihood': level_fit.llf, 'aic': level_fit.aic, 'bic': level_fit.bic,
     'lr_vs_additive': lr_statistic, 'df': lr_df,
     'p_value': chi2.sf(lr_statistic, lr_df)},
])
print('MODEL COMPARISON: does adding all interactions improve fit?')
display(level_model_comparison.round(4))
print('INTERACTION MODEL')
print(level_fit.summary())

level_coefficient_table = clean_fixed_effect_table(level_fit)
display(level_coefficient_table.round(4))

In [ ]:
level_fe_names = list(level_fit.fe_params.index)
level_fe_values = level_fit.fe_params.to_numpy(float)
level_fe_cov = level_fit.cov_params().loc[level_fe_names, level_fe_names].to_numpy(float)
level_fixed_formula = level_formula.split('~', 1)[1].strip()

def level_design_row(condition, retention_status, source_age):
    frame = pd.DataFrame({
        'condition': pd.Categorical([condition], categories=['actual', 'younger', 'older']),
        'retention_status': pd.Categorical(
            [retention_status], categories=['omitted', 'retained']
        ),
        'source_age_centered': [source_age - AGE_CENTER],
    })
    matrix = dmatrix(level_fixed_formula, frame, return_type='dataframe')
    return matrix.reindex(columns=level_fe_names, fill_value=0.0).iloc[0].to_numpy(float)

def level_linear_test(name, contrast):
    contrast = np.asarray(contrast, float)
    estimate = float(contrast @ level_fe_values)
    se = float(np.sqrt(max(contrast @ level_fe_cov @ contrast, 0)))
    z = estimate / se if se > 0 else np.nan
    p_value = 2 * norm.sf(abs(z)) if np.isfinite(z) else np.nan
    return {'test': name, 'estimate': estimate, 'se': se, 'z': z,
            'p_value': p_value, 'ci95_low': estimate - 1.96 * se,
            'ci95_high': estimate + 1.96 * se}

trajectory_tests = []
for condition in CONDITION_ORDER:
    for status in ['omitted', 'retained']:
        slope = (level_design_row(condition, status, AGE_CENTER + 0.5)
                 - level_design_row(condition, status, AGE_CENTER - 0.5))
        row = level_linear_test(
            f'{condition}, {status}: source-age slope per year', slope
        )
        row.update({'condition': condition, 'status': status, 'contrast': 'age_slope'})
        trajectory_tests.append(row)
    slope_difference = (
        level_design_row(condition, 'retained', AGE_CENTER + 0.5)
        - level_design_row(condition, 'retained', AGE_CENTER - 0.5)
        - level_design_row(condition, 'omitted', AGE_CENTER + 0.5)
        + level_design_row(condition, 'omitted', AGE_CENTER - 0.5)
    )
    row = level_linear_test(
        f'{condition}: retained minus omitted slope difference', slope_difference
    )
    row.update({'condition': condition, 'status': 'retained - omitted',
                'contrast': 'slope_difference'})
    trajectory_tests.append(row)
trajectory_results = pd.DataFrame(trajectory_tests)
p_values = trajectory_results.p_value.to_numpy(float)
valid = np.flatnonzero(np.isfinite(p_values))
order = valid[np.argsort(p_values[valid])]
holm_sorted = np.maximum.accumulate(
    (len(order) - np.arange(len(order))) * p_values[order]
) if len(order) else np.asarray([])
trajectory_results['p_holm'] = np.nan
trajectory_results.loc[order, 'p_holm'] = np.minimum(holm_sorted, 1.0)
trajectory_results['significant_raw_05'] = trajectory_results.p_value < 0.05
trajectory_results['significant_holm_05'] = trajectory_results.p_holm < 0.05
display(trajectory_results[[
    'condition', 'status', 'contrast', 'estimate', 'se', 'ci95_low', 'ci95_high',
    'z', 'p_value', 'p_holm', 'significant_raw_05', 'significant_holm_05'
]].round(4))

level_support = level_data.groupby('condition', observed=True)['source_age'].agg(['min', 'max'])
level_age_min = float(level_support['min'].max())
level_age_max = float(level_support['max'].min())
level_age_grid = np.linspace(level_age_min, level_age_max, 100)
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
status_colors = {'omitted': '#E15759', 'retained': '#4C78A8'}
for ax, condition in zip(axes, CONDITION_ORDER):
    for status in ['omitted', 'retained']:
        estimates, lower, upper = [], [], []
        for age in level_age_grid:
            x = level_design_row(condition, status, age)
            estimate = float(x @ level_fe_values)
            se = float(np.sqrt(max(x @ level_fe_cov @ x, 0)))
            estimates.append(estimate)
            lower.append(estimate - 1.96 * se)
            upper.append(estimate + 1.96 * se)
        ax.plot(level_age_grid, estimates, color=status_colors[status], label=status)
        ax.fill_between(level_age_grid, lower, upper, color=status_colors[status], alpha=.16)
    ax.set(title=condition, xlabel='Source age')
axes[0].set_ylabel('Fitted mean original-detail centrality')
axes[-1].legend(title='Detail status')
fig.suptitle('Retained and omitted centrality trajectories')
plt.tight_layout()
plt.show()

## RQ6 — Confirmatory centrality-gap regression and hypothesis tests

Fit the random-intercept model

```r
centrality_gap ~ source_age * condition + (1 | story)
```

with `actual` as the reference condition and source age centered at 8.

Hypotheses:

1. The fitted centrality gap differs from zero within each condition at source age 8.
2. The source-age slope differs from zero within each condition.
3. Younger, actual, and older summaries differ at the same source age.
4. Source-age slopes differ by condition.

Same-age contrasts are evaluated at three representative ages inside the common-support range. Holm adjustment covers those pairwise condition tests.

In [ ]:
try:
    import statsmodels.formula.api as smf
    from patsy import dmatrix
    from scipy.stats import norm
except ImportError as error:
    raise ImportError(
        "The final mixed model requires statsmodels, patsy, and scipy. "
        "Run this notebook inside the project Singularity environment."
    ) from error

model_data["condition"] = pd.Categorical(
    model_data["condition"], categories=["actual", "younger", "older"]
)
formula = (
    'centrality_gap ~ source_age_centered * '
    'C(condition, Treatment(reference="actual"))'
)
mixed = smf.mixedlm(
    formula, data=model_data, groups=model_data["story_key"], re_formula="1"
)
fit = mixed.fit(reml=False, method="lbfgs")
print(fit.summary())

In [ ]:
fe_names = list(fit.fe_params.index)
fe_values = fit.fe_params.to_numpy(float)
fe_cov = fit.cov_params().loc[fe_names, fe_names].to_numpy(float)
fixed_effect_formula = formula.split('~', 1)[1].strip()

def design_row(condition, source_age):
    frame = pd.DataFrame({
        "condition": pd.Categorical(
            [condition], categories=["actual", "younger", "older"]
        ),
        "source_age_centered": [source_age - AGE_CENTER],
    })
    matrix = dmatrix(fixed_effect_formula, frame, return_type='dataframe')
    matrix = matrix.reindex(columns=fe_names, fill_value=0.0)
    return matrix.iloc[0].to_numpy(float)

def linear_test(name, contrast):
    contrast = np.asarray(contrast, float)
    estimate = float(contrast @ fe_values)
    se = float(np.sqrt(max(contrast @ fe_cov @ contrast, 0)))
    z = estimate / se if se > 0 else np.nan
    p_value = 2 * norm.sf(abs(z)) if np.isfinite(z) else np.nan
    return {
        "test": name, "estimate": estimate, "se": se,
        "z": z, "p_value": p_value,
    }

tests = []
for condition in CONDITION_ORDER:
    tests.append(linear_test(
        f"{condition}: fitted gap at source age 8",
        design_row(condition, AGE_CENTER),
    ))
    slope = (
        design_row(condition, AGE_CENTER + 0.5)
        - design_row(condition, AGE_CENTER - 0.5)
    )
    tests.append(linear_test(
        f"{condition}: source-age slope per year", slope
    ))

display(pd.DataFrame(tests))

In [ ]:
support = model_data.groupby(
    "condition", observed=True
)["source_age"].agg(["min", "max"])
common_min = float(support["min"].max())
common_max = float(support["max"].min())
if common_min > common_max:
    raise ValueError("No common source-age support across all three conditions.")

age_sets = [
    set(model_data.loc[model_data["condition"].eq(condition), "source_age"].dropna())
    for condition in CONDITION_ORDER
]
common_ages = sorted(set.intersection(*age_sets))
if not common_ages:
    raise ValueError("No exactly observed source ages are shared by all conditions.")
age_positions = np.unique(np.round(np.linspace(0, len(common_ages) - 1, min(3, len(common_ages)))).astype(int))
comparison_ages = [float(common_ages[position]) for position in age_positions]
pairs = [
    ("younger", "actual"),
    ("younger", "older"),
    ("actual", "older"),
]
contrasts = []
for age in comparison_ages:
    for left, right in pairs:
        row = linear_test(
            f"{left} - {right} at source age {age:g}",
            design_row(left, age) - design_row(right, age),
        )
        row.update({"source_age": age, "left": left, "right": right})
        contrasts.append(row)

contrast_frame = pd.DataFrame(contrasts)
order = np.argsort(contrast_frame["p_value"].to_numpy())
adjusted_sorted = np.maximum.accumulate(
    (len(order) - np.arange(len(order)))
    * contrast_frame["p_value"].to_numpy()[order]
)
adjusted = np.empty(len(order))
adjusted[order] = np.minimum(adjusted_sorted, 1.0)
contrast_frame["p_holm"] = adjusted

display(support)
print("Representative common-support ages:", comparison_ages)
display(contrast_frame[[
    "test", "estimate", "se", "z", "p_value", "p_holm"
]])

age_grid = np.linspace(common_min, common_max, 100)
fig, ax = plt.subplots(figsize=(9, 5))
for condition in CONDITION_ORDER:
    estimates, lower, upper = [], [], []
    for age in age_grid:
        x = design_row(condition, age)
        estimate = float(x @ fe_values)
        se = float(np.sqrt(max(x @ fe_cov @ x, 0)))
        estimates.append(estimate)
        lower.append(estimate - 1.96 * se)
        upper.append(estimate + 1.96 * se)
    ax.plot(age_grid, estimates, label=condition, color=colors[condition])
    ax.fill_between(
        age_grid, lower, upper, color=colors[condition], alpha=.18
    )
ax.axhline(0, color="black", linewidth=1)
ax.set(
    xlabel="Source age",
    ylabel="Fitted retained − omitted centrality",
    title="Mixed-model centrality selection within common age support",
)
ax.legend(title="Summary condition")
plt.show()

## RQ7 — Within stories having all three summaries, how large are the paired condition effects?

This complete-triplet sensitivity analysis retains only stories observed in `younger`, `actual`, and `older`. For each condition pair it reports the mean within-story difference, a paired *t* test, a 95% CI, and paired Cohen's $d_z=\bar{D}/s_D$. Holm correction covers the three paired condition tests. Unlike the mixed model above, this section estimates condition effects only in the smaller complete-triplet subset and does not model source-age moderation.

In [ ]:
from scipy.stats import t as student_t, ttest_1samp

story_condition = model_data.groupby(
    ['story_key', 'condition'], observed=True, as_index=False
).agg(centrality_gap=('centrality_gap', 'mean'), source_age=('source_age', 'first'))
triplets = story_condition.pivot(
    index='story_key', columns='condition', values='centrality_gap'
).reindex(columns=CONDITION_ORDER).dropna()
print(f'Complete three-condition stories: {len(triplets):,}')

paired_conditions = [('younger', 'actual'), ('older', 'actual'), ('older', 'younger')]
paired_rows = []
for left, right in paired_conditions:
    difference = (triplets[left] - triplets[right]).dropna().to_numpy(float)
    n = len(difference)
    mean_difference = float(np.mean(difference))
    sd_difference = float(np.std(difference, ddof=1)) if n > 1 else np.nan
    se_difference = sd_difference / np.sqrt(n) if n > 1 else np.nan
    critical = float(student_t.ppf(0.975, n - 1)) if n > 1 else np.nan
    test = ttest_1samp(difference, 0.0) if n > 1 else None
    paired_rows.append({
        'contrast': f'{left} - {right}', 'stories': n,
        'mean_paired_difference': mean_difference,
        'ci95_low': mean_difference - critical * se_difference,
        'ci95_high': mean_difference + critical * se_difference,
        'cohens_dz': mean_difference / sd_difference if sd_difference > 0 else np.nan,
        't': float(test.statistic) if test is not None else np.nan,
        'p_value': float(test.pvalue) if test is not None else np.nan,
    })
paired_effects = pd.DataFrame(paired_rows)
order = np.argsort(paired_effects.p_value.to_numpy())
holm_sorted = np.maximum.accumulate(
    (len(order) - np.arange(len(order))) * paired_effects.p_value.to_numpy()[order]
)
paired_effects['p_holm'] = np.nan
paired_effects.loc[order, 'p_holm'] = np.minimum(holm_sorted, 1.0)
display(paired_effects)

fig, ax = plt.subplots(figsize=(7, 5))
x = np.arange(len(CONDITION_ORDER))
for _, row in triplets.iterrows():
    ax.plot(x, row.to_numpy(float), color='grey', alpha=0.10, linewidth=0.7)
ax.plot(x, triplets.mean().to_numpy(float), color='black', marker='o', linewidth=2.5,
        label='Mean')
ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
ax.set_xticks(x, CONDITION_ORDER)
ax.set(xlabel='Summary condition', ylabel='Retained − omitted centrality',
       title='Complete-triplet within-story centrality gaps')
ax.legend()
plt.show()